# Tutorial: Introduction to JAX

## What You Will Learn

This tutorial provides a step-by-step introduction to **JAX**, a numerical computing library developed by Google. By the end, you will understand:

1. **What JAX is** and why it's useful for machine learning
2. **How to use JAX like NumPy** (it's almost identical!)
3. **How to compute gradients automatically** using `jax.grad`
4. **How to vectorize functions** using `jax.vmap`
5. **How JAX handles random numbers** (this is different from NumPy!)
6. **How to speed up your code** using `jax.jit`

## Prerequisites

You should be familiar with:
- Basic Python programming
- NumPy basics (arrays, operations like `np.sin`, `np.sum`, etc.)
- Basic calculus (what a derivative is)

Don't worry if you're rusty on any of these - we'll explain everything as we go!

## 1. What is JAX and Why Should You Care?

### The Problem JAX Solves

In machine learning, we often need to:
1. **Compute gradients** (derivatives) of functions - this is essential for training neural networks
2. **Run computations on GPUs** for speed
3. **Process batches of data** efficiently

Traditionally, you would need different tools for each of these tasks. **JAX combines all of them into one library** with a familiar NumPy-like interface.

### JAX = NumPy + Superpowers

Think of JAX as NumPy with three extra abilities:

| Superpower | What it does | JAX function |
|------------|--------------|--------------|
| **Automatic Differentiation** | Computes derivatives for you | `jax.grad()` |
| **Automatic Vectorization** | Makes functions work on batches | `jax.vmap()` |
| **JIT Compilation** | Makes code run faster on CPU/GPU/TPU | `jax.jit()` |

Let's see each of these in action!

## 2. Getting Started: JAX as NumPy

The first thing to understand is that **JAX looks almost exactly like NumPy**. If you know NumPy, you already know 90% of JAX!

### 2.1 The One Change You Need to Make

Instead of writing:
```python
import numpy as np
```

You write:
```python
import jax.numpy as jnp
```

That's it! Most of your NumPy code will work with just this change.

Let's start by importing the libraries we need:

In [ ]:
# First, let's import JAX and NumPy so we can compare them
import jax  # The main JAX library
import jax.numpy as jnp  # JAX's version of NumPy (we use 'jnp' by convention)
import numpy as np  # Regular NumPy (we use 'np' by convention)
import matplotlib.pyplot as plt  # For plotting

# Let's check that everything imported correctly
print("JAX version:", jax.__version__)
print("NumPy version:", np.__version__)

### 2.2 Creating Arrays: NumPy vs JAX

Let's create the same array in both NumPy and JAX to see how similar they are:

In [ ]:
# Creating an array in NumPy
numpy_array = np.array([1.0, 2.0, 3.0])
print("NumPy array:", numpy_array)
print("Type:", type(numpy_array))

In [ ]:
# Creating the SAME array in JAX - notice the syntax is identical!
jax_array = jnp.array([1.0, 2.0, 3.0])
print("JAX array:", jax_array)
print("Type:", type(jax_array))

**Key observation**: The syntax is exactly the same! The only difference is:
- NumPy arrays have type `numpy.ndarray`
- JAX arrays have type `jax.Array`

JAX arrays can live on different devices (CPU, GPU, or TPU), which is why they have a different type.

### 2.3 Common Operations Work the Same Way

All the NumPy functions you know have JAX equivalents. Let's try some:

#### Creating Special Arrays

In [ ]:
# Creating a 3x3 matrix of zeros
zeros = jnp.zeros((3, 3))
print("A 3x3 matrix of zeros:")
print(zeros)

In [ ]:
# Creating a 2x4 matrix of ones
ones = jnp.ones((2, 4))
print("A 2x4 matrix of ones:")
print(ones)

In [ ]:
# Creating a 3x3 identity matrix (1s on diagonal, 0s elsewhere)
identity = jnp.eye(3)
print("A 3x3 identity matrix:")
print(identity)

#### Mathematical Operations

In [ ]:
# Let's create an array and apply some math functions
x = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
print("Our array x:", x)

In [ ]:
# Sine function (element-wise)
print("sin(x) =", jnp.sin(x))

In [ ]:
# Exponential function (element-wise)
print("exp(x) =", jnp.exp(x))

In [ ]:
# Sum of all elements
print("sum(x) =", jnp.sum(x))

In [ ]:
# Mean (average) of all elements
print("mean(x) =", jnp.mean(x))

#### Matrix Operations

In [ ]:
# Let's create two 2x2 matrices
A = jnp.array([[1.0, 2.0], [3.0, 4.0]])

B = jnp.array([[5.0, 6.0], [7.0, 8.0]])

print("Matrix A:")
print(A)
print("\nMatrix B:")
print(B)

In [ ]:
# Matrix multiplication: A @ B
# This computes the standard matrix product
print("Matrix multiplication (A @ B):")
print(A @ B)

In [ ]:
# Element-wise multiplication: A * B
# This multiplies corresponding elements
print("Element-wise multiplication (A * B):")
print(A * B)

In [ ]:
# Determinant of a matrix
print("Determinant of A:", jnp.linalg.det(A))

### 2.4 One Important Difference: JAX Arrays Are Immutable

Here's where JAX differs from NumPy. In NumPy, you can modify an array in place:

```python
# This works in NumPy
x = np.array([1, 2, 3])
x[0] = 10  # Modify the first element
print(x)   # Output: [10, 2, 3]
```

**In JAX, arrays cannot be modified after creation.** They are "immutable" (unchangeable).

Why? Because JAX needs this property to:
1. Compute gradients correctly
2. Compile code for GPUs/TPUs
3. Parallelize computations

Let's see how to work with this:

In [ ]:
# Create an array
x = jnp.array([1, 2, 3])
print("Original x:", x)

In [ ]:
# If you try x[0] = 10, you'll get an error!
# Instead, use the .at[].set() method:

# This creates a NEW array with the modification
x_modified = x.at[0].set(10)

print("Modified array:", x_modified)
print("Original array (unchanged!):", x)

**Key point**: The `.at[].set()` method doesn't change the original array. Instead, it creates a brand new array with the modification applied. The original array stays the same.

This might seem inconvenient at first, but it's actually very useful for:
- Debugging (you can always check the original values)
- Parallel computing (no conflicts when multiple operations happen at once)
- Computing gradients (JAX can track all operations)

In [ ]:
# You can chain multiple modifications:
x = jnp.array([1, 2, 3, 4, 5])
print("Original:", x)

# Change element at index 0 to 100, and element at index 2 to 300
x_new = x.at[0].set(100).at[2].set(300)
print("After modifications:", x_new)

## 3. Automatic Differentiation with `jax.grad`

This is JAX's first superpower and perhaps the most important one for machine learning!

### 3.1 What is a Gradient?

If you have a function $f(x)$, its **gradient** (or derivative) $f'(x)$ tells you how much $f$ changes when you slightly change $x$.

For example:
- If $f(x) = x^2$, then $f'(x) = 2x$
- If $f(x) = \sin(x)$, then $f'(x) = \cos(x)$

In machine learning, we use gradients to find the minimum of a loss function (this is called "gradient descent").

### 3.2 The Problem: Computing Gradients is Tedious

Normally, to get the gradient, you have to:
1. Work out the derivative formula by hand (using calculus rules)
2. Implement that formula in code

This is error-prone and time-consuming, especially for complex functions.

### 3.3 The Solution: `jax.grad` Does It For You!

`jax.grad` takes any Python function and **automatically creates a new function** that computes the gradient.

Let's see this with a simple example:

In [ ]:
# Step 1: Define a simple function
# Let's use f(x) = sin(x) + 0.1*x
def f(x):
    return jnp.sin(x) + 0.1 * x


# Let's verify what this function looks like at a few points
print("f(0) =", f(0.0))
print("f(1) =", f(1.0))
print("f(2) =", f(2.0))

Now, let's compute the derivative. From calculus, we know:
- If $f(x) = \sin(x) + 0.1x$
- Then $f'(x) = \cos(x) + 0.1$

Let's verify this with JAX:

In [ ]:
# Step 2: Create the gradient function using jax.grad
# jax.grad takes a function as input and returns a NEW function
f_prime = jax.grad(f)

# f_prime is now a function that computes the derivative of f!
print("Type of f_prime:", type(f_prime))

In [ ]:
# Step 3: Use the gradient function
# Let's compute f'(1.0)

x = 1.0

# Manual calculation: f'(x) = cos(x) + 0.1
manual_derivative = jnp.cos(x) + 0.1
print("Manual derivative f'(1.0) = cos(1.0) + 0.1 =", manual_derivative)

# JAX automatic calculation
jax_derivative = f_prime(x)
print("JAX derivative f'(1.0) =", jax_derivative)

# They should match!
print("Do they match?", jnp.isclose(manual_derivative, jax_derivative))

**Amazing!** JAX computed the derivative automatically, without us having to figure out the formula!

Let's break down what happened:
1. We defined `f(x) = sin(x) + 0.1*x`
2. We called `jax.grad(f)` which returned a new function `f_prime`
3. `f_prime(x)` computes $f'(x) = \cos(x) + 0.1$ for any value of $x$

### 3.4 Visualizing the Function and its Gradient

Let's plot both the function and its gradient to see how they relate:

In [ ]:
# Create many x values from -5 to 5
x_values = jnp.linspace(-5, 5, 100)  # 100 points between -5 and 5

# Compute f(x) for all these points
# Note: f expects a single number, but we want to apply it to many numbers
# We'll use jax.vmap for this (explained in detail later!)
f_values = jax.vmap(f)(x_values)

# Compute f'(x) for all these points
gradient_values = jax.vmap(f_prime)(x_values)

In [ ]:
# Now let's plot!
fig, ax = plt.subplots(figsize=(10, 5))

# Plot the original function
ax.plot(x_values, f_values, label=r"$f(x) = \sin(x) + 0.1x$", linewidth=2, color="blue")

# Plot the gradient
ax.plot(
    x_values,
    gradient_values,
    label=r"$f'(x) = \cos(x) + 0.1$",
    linewidth=2,
    color="red",
    linestyle="--",
)

# Add a horizontal line at y=0 for reference
ax.axhline(y=0, color="gray", linestyle=":", alpha=0.5)

# Add labels and legend
ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("y", fontsize=12)
ax.set_title(
    "Function and its Gradient (computed automatically with JAX!)", fontsize=14
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.show()

**What to notice in the plot:**
- When $f'(x) > 0$ (red line above zero), $f(x)$ is increasing (blue line going up)
- When $f'(x) < 0$ (red line below zero), $f(x)$ is decreasing (blue line going down)
- When $f'(x) = 0$ (red line crosses zero), $f(x)$ has a local maximum or minimum

### 3.5 Getting Both Value and Gradient: `jax.value_and_grad`

In machine learning, we often need both:
- The **value** of a function (e.g., the loss)
- The **gradient** of that function (to update parameters)

We could compute them separately:
```python
value = f(x)           # One computation
gradient = f_prime(x)  # Another computation
```

But this is inefficient! JAX provides `jax.value_and_grad` which computes both at once:

In [ ]:
# Create a function that returns BOTH the value and the gradient
f_value_and_grad = jax.value_and_grad(f)

# Now when we call it, we get a tuple: (value, gradient)
x = 1.0
result = f_value_and_grad(x)

print("Result type:", type(result))
print("Result:", result)

In [ ]:
# We can unpack the tuple into two variables:
value, gradient = f_value_and_grad(x)

print(f"At x = {x}:")
print(f"  Function value f(x) = {value}")
print(f"  Gradient f'(x) = {gradient}")

# Verify these are correct:
print(f"\n  Check: f({x}) = sin({x}) + 0.1*{x} = {jnp.sin(x) + 0.1 * x}")
print(f"  Check: f'({x}) = cos({x}) + 0.1 = {jnp.cos(x) + 0.1}")

### 3.6 Gradients of Functions with Multiple Inputs

What if your function has multiple inputs? For example:

$$g(x, y) = x^2 + 3xy + y^2$$

This function has **two** variables, so it has **two** partial derivatives:
- $\frac{\partial g}{\partial x} = 2x + 3y$ (how $g$ changes when $x$ changes)
- $\frac{\partial g}{\partial y} = 3x + 2y$ (how $g$ changes when $y$ changes)

By default, `jax.grad` computes the derivative with respect to the **first argument**. You can change this using `argnums`:

In [ ]:
# Define a function with two inputs
def g(x, y):
    return x**2 + 3 * x * y + y**2


# Test the function
print("g(2, 3) =", g(2.0, 3.0))
print("Manual check: 2² + 3*2*3 + 3² = 4 + 18 + 9 =", 4 + 18 + 9)

In [ ]:
# Gradient with respect to the FIRST argument (x)
# This is the default behavior
dg_dx = jax.grad(g, argnums=0)  # argnums=0 means "first argument"

x, y = 2.0, 3.0
print(f"∂g/∂x at (x={x}, y={y}):")
print(f"  JAX result: {dg_dx(x, y)}")
print(f"  Manual: 2x + 3y = 2*{x} + 3*{y} = {2 * x + 3 * y}")

In [ ]:
# Gradient with respect to the SECOND argument (y)
dg_dy = jax.grad(g, argnums=1)  # argnums=1 means "second argument"

print(f"\n∂g/∂y at (x={x}, y={y}):")
print(f"  JAX result: {dg_dy(x, y)}")
print(f"  Manual: 3x + 2y = 3*{x} + 2*{y} = {3 * x + 2 * y}")

In [ ]:
# Gradient with respect to BOTH arguments at once
dg_dxy = jax.grad(g, argnums=(0, 1))  # Both arguments

result = dg_dxy(x, y)
print(f"\nBoth gradients at (x={x}, y={y}):")
print(f"  Result: {result}")
print(f"  This is a tuple: (∂g/∂x, ∂g/∂y) = ({result[0]}, {result[1]})")

### 3.7 Gradients with Dictionaries (Important for Machine Learning!)

In machine learning, we often have multiple parameters organized in a dictionary:

```python
params = {"weight": 2.0, "bias": 0.5}
```

JAX can compute gradients with respect to **all parameters in the dictionary at once**! This is extremely useful for training neural networks.

In [ ]:
# Let's create a simple linear model: prediction = a*x + b
# We want to find the best values of 'a' and 'b'


def loss_function(params, x, y):
    """
    Compute the squared error loss.

    params: dictionary with 'a' and 'b'
    x: input value
    y: true output value

    Returns: (prediction - true_value)^2
    """
    # Make a prediction using our model
    prediction = params["a"] * x + params["b"]

    # Compute squared error
    loss = (prediction - y) ** 2

    return loss

In [ ]:
# Set up our parameters and data
params = {"a": 1.0, "b": 0.0}  # Initial guess: y = 1*x + 0
x = 2.0  # Input
y = 5.0  # True output (we want our model to predict 5 when x=2)

# What does our model predict?
prediction = params["a"] * x + params["b"]
print(f"Our model: y = {params['a']}*x + {params['b']}")
print(f"For x = {x}, we predict: {prediction}")
print(f"True value: {y}")
print(f"Loss (squared error): {loss_function(params, x, y)}")

In [ ]:
# Now compute gradients with respect to params
# This tells us how to change 'a' and 'b' to reduce the loss

grad_fn = jax.grad(loss_function, argnums=0)  # Gradient w.r.t. first argument (params)
gradients = grad_fn(params, x, y)

print("\nGradients:")
print(f"  ∂loss/∂a = {gradients['a']}")
print(f"  ∂loss/∂b = {gradients['b']}")

**How to interpret these gradients:**
- `∂loss/∂a = -12.0` means: if we increase `a` by a tiny amount, the loss decreases
- `∂loss/∂b = -6.0` means: if we increase `b` by a tiny amount, the loss decreases

To minimize the loss, we should move in the **opposite direction** of the gradient (this is gradient descent!)

In [ ]:
# Let's manually do one step of gradient descent
learning_rate = 0.1

new_a = params["a"] - learning_rate * gradients["a"]
new_b = params["b"] - learning_rate * gradients["b"]

print(f"Before: a = {params['a']}, b = {params['b']}")
print(f"After:  a = {new_a}, b = {new_b}")
print(f"\nOld prediction: {params['a'] * x + params['b']}")
print(f"New prediction: {new_a * x + new_b}")
print(f"True value: {y}")
print("The new prediction is closer to the true value!")

## 4. Automatic Vectorization with `jax.vmap`

This is JAX's second superpower! `jax.vmap` automatically makes functions work on **batches** of data.

### 4.1 The Problem: Processing Multiple Inputs

Suppose you write a function that works on a single input:

In [ ]:
def process_single_vector(x, weights):
    """
    Process a single vector x with some weights.
    This is a simple dot product followed by squaring.

    x: a 1D array (vector)
    weights: a 1D array of same length as x
    """
    result = jnp.dot(x, weights)  # Dot product
    return result**2  # Square the result


# Test with a single vector
x = jnp.array([1.0, 2.0, 3.0])
w = jnp.array([0.5, 0.5, 0.5])

result = process_single_vector(x, w)
print("Input x:", x)
print("Weights:", w)
print("Dot product:", jnp.dot(x, w))
print("Squared:", result)

Now, what if we have **many vectors** to process? This is common in machine learning where we process batches of data.

In [ ]:
# Create a batch of 4 vectors (each vector has 3 elements)
X_batch = jnp.array(
    [
        [1.0, 2.0, 3.0],  # First vector
        [4.0, 5.0, 6.0],  # Second vector
        [7.0, 8.0, 9.0],  # Third vector
        [0.0, 1.0, 0.0],  # Fourth vector
    ]
)

print("Batch of vectors (shape: 4 vectors × 3 elements):")
print(X_batch)
print("Shape:", X_batch.shape)

### 4.2 The Naive Solution: Use a Loop

The obvious approach is to loop over each vector:

In [ ]:
# Process each vector one at a time with a loop
results_loop = []
for i in range(len(X_batch)):
    result = process_single_vector(X_batch[i], w)
    results_loop.append(result)
results_loop = jnp.array(results_loop)

print("Results using loop:", results_loop)

This works, but it's **slow**! Python loops are not efficient for numerical computation.

### 4.3 The JAX Solution: Use `jax.vmap`

`jax.vmap` (Vectorized MAP) takes a function that works on single inputs and **automatically** creates a version that works on batches:

In [ ]:
# Create a "batched" version of our function
# vmap says: "apply this function to each row of the input"
batched_process = jax.vmap(process_single_vector, in_axes=(0, None))
#                                                 ^^^^^^^^^^^^^^
#                           in_axes tells vmap which arguments to batch over:
#                           - 0 means "batch over the first axis (rows) of this argument"
#                           - None means "don't batch this argument, use it as-is"

# Now we can process the whole batch at once!
results_vmap = batched_process(X_batch, w)

print("Results using vmap:", results_vmap)
print("\nSame as loop results?", jnp.allclose(results_loop, results_vmap))

### 4.4 Understanding `in_axes`

The `in_axes` argument tells `vmap` how to batch each input:

| `in_axes` value | Meaning |
|-----------------|---------|
| `0` | Batch over the first axis (rows) |
| `1` | Batch over the second axis (columns) |
| `None` | Don't batch this argument; use it for all iterations |

Let's see more examples:

In [ ]:
# Example 1: Batch BOTH arguments
# Different weights for each vector
W_batch = jnp.array(
    [
        [0.5, 0.5, 0.5],  # Weights for first vector
        [1.0, 0.0, 0.0],  # Weights for second vector
        [0.0, 1.0, 0.0],  # Weights for third vector
        [0.0, 0.0, 1.0],  # Weights for fourth vector
    ]
)

# Batch both: in_axes=(0, 0)
batched_both = jax.vmap(process_single_vector, in_axes=(0, 0))
results = batched_both(X_batch, W_batch)

print("X_batch (4 vectors):")
print(X_batch)
print("\nW_batch (4 different weights):")
print(W_batch)
print("\nResults (each vector processed with its own weights):")
print(results)

In [ ]:
# Example 2: Same weights for all vectors
# in_axes=(0, None) - batch X, but use same w for all

batched_same_w = jax.vmap(process_single_vector, in_axes=(0, None))
results = batched_same_w(X_batch, w)

print("Same weights w =", w, "for all vectors")
print("Results:", results)

### 4.5 Using `vmap` as a Decorator

Instead of writing `batched_fn = jax.vmap(fn, ...)`, you can use `vmap` as a **decorator**. This makes the code cleaner when you always want the batched version.

To use `vmap` as a decorator with arguments like `in_axes`, we need `functools.partial`:

In [ ]:
from functools import partial


# Without decorator (what we did before):
def my_function_v1(x, w):
    return jnp.dot(x, w) ** 2


batched_v1 = jax.vmap(my_function_v1, in_axes=(0, None))

In [ ]:
# With decorator (cleaner!):
@partial(jax.vmap, in_axes=(0, None))
def my_function_v2(x, w):
    return jnp.dot(x, w) ** 2


# my_function_v2 is ALREADY the batched version!

In [ ]:
# Both work the same way:
print("Using regular vmap:")
print(batched_v1(X_batch, w))

print("\nUsing decorator:")
print(my_function_v2(X_batch, w))

**When to use the decorator style:**
- When you always want the batched version of the function
- When you want cleaner code without a separate `batched_fn = jax.vmap(fn)` line

**When to use the regular style:**
- When you sometimes need the single-input version
- When you want to be more explicit about what's happening

## 5. Random Number Generation in JAX

JAX handles random numbers **differently** from NumPy. This section is important to understand!

### 5.1 How NumPy Does Randomness (The Problem)

In NumPy, there's a **global** random state that changes every time you generate a random number:

```python

import numpy as np
np.random.seed(42)     # Set the global state
x = np.random.normal() # Generate number, state changes
y = np.random.normal() # Generate different number, state changes again
```

This causes problems:
1. **Reproducibility is tricky**: The global state changes silently
2. **Parallelization is hard**: Multiple threads sharing one state = chaos
3. **Debugging is difficult**: Hard to reproduce exact same random sequence

### 5.2 How JAX Does Randomness (The Solution)

In JAX, you **explicitly manage** random keys. Think of a key as a "seed" that you pass around:

In [ ]:
# Step 1: Create a random key (like setting a seed)
key = jax.random.key(42)  # 42 is the seed
print("Random key:", key)
print("Type:", type(key))

A key is just an array of numbers that determines the random sequence. Now let's use it:

In [ ]:
# Step 2: Generate a random number using the key
random_number = jax.random.uniform(key)  # Uniform between 0 and 1
print("Random number:", random_number)

In [ ]:
# IMPORTANT: Using the SAME key gives the SAME number!
random_number_1 = jax.random.uniform(key)
random_number_2 = jax.random.uniform(key)
print("First call with key:", random_number_1)
print("Second call with same key:", random_number_2)
print("Are they equal?", random_number_1 == random_number_2)

**This is intentional!** Same key = same result. This makes JAX **100% reproducible**.

But wait... if the same key always gives the same number, how do we get different random numbers?

### 5.3 Splitting Keys to Get New Random Numbers

The answer is: **split the key** to create new, different keys.

Think of it like this:
- Original key → split → two new keys (that are different from each other AND from the original)

The **golden rule**: Never reuse a key. Always split before generating new random numbers.

In [ ]:
# Start with a key
key = jax.random.key(0)
print("Original key:", key)

In [ ]:
# Split the key into TWO new keys
key, subkey = jax.random.split(key)
#  ^-- save this for future splits   ^-- use this for random operation

print("After splitting:")
print("  key (for future splits):", key)
print("  subkey (for this random operation):", subkey)

In [ ]:
# Use subkey to generate a random number
x = jax.random.normal(subkey)  # Standard normal distribution
print("\nRandom number from subkey:", x)

In [ ]:
# To get another random number, split again!
key, subkey = jax.random.split(key)
y = jax.random.normal(subkey)
print("Another random number:", y)

# And again...
key, subkey = jax.random.split(key)
z = jax.random.normal(subkey)
print("Yet another random number:", z)

**The pattern to remember:**
```python
key = jax.random.PRNGKey(seed)      # Initial key

key, subkey = jax.random.split(key) # Split before each use
result1 = jax.random.normal(subkey) # Use subkey

key, subkey = jax.random.split(key) # Split again
result2 = jax.random.normal(subkey) # Use new subkey
```

### 5.4 Generating Multiple Keys at Once

You can split a key into **many** keys at once:

In [ ]:
key = jax.random.key(42)

# Split into 5 different keys at once
keys = jax.random.split(key, num=5)

print("5 different keys:")
for i, k in enumerate(keys):
    print(f"  Key {i}: {k}")

In [ ]:
# Use each key to generate a different random number
samples = []
for k in keys:
    sample = jax.random.normal(k)
    samples.append(sample)
samples = jnp.array(samples)

print("5 random samples:", samples)

### 5.5 Combining `vmap` with Random Numbers

Here's where JAX's explicit key system really shines! We can use `vmap` to generate many random numbers in parallel.

Remember: `vmap` applies a function to each element of an array. If we have an array of keys, `vmap` will apply a random function to each key, giving us different random numbers!

In [ ]:
# Define a function that takes a key and returns a random operation
def sample_and_square(key):
    """Generate a random number and return its square."""
    x = jax.random.uniform(key)  # Random number between 0 and 1
    return x**2  # Square it


# Test with one key
key = jax.random.PRNGKey(0)
result = sample_and_square(key)
print("Single result:", result)

In [ ]:
# Now let's do this for 10 different random numbers using vmap!

# Step 1: Generate 10 different keys
keys = jax.random.split(key, 10)
print("10 different keys:")
print(keys)

In [ ]:
# Step 2: Apply sample_and_square to each key using vmap
results = jax.vmap(sample_and_square)(keys)

print("\n10 different random samples squared:")
print(results)

This is much more efficient than a Python loop, especially on GPUs!

### 5.6 Exercise: Combining `vmap` and Random Keys

**Task**: Write a function that:
1. Takes a random key and a constant `k`
2. Generates a Gaussian (normal) random number
3. Adds the constant `k` to it
4. Returns the result

Then use `jax.vmap` to generate 10 random numbers, all with the same constant added.

In [ ]:
# Solution:
# We use the decorator style with in_axes=(0, None)
# - 0: batch over the first argument (keys)
# - None: don't batch the second argument (use same k for all)


@partial(jax.vmap, in_axes=(0, None))
def add_constant_to_random(key, k):
    """
    Generate a Gaussian random number and add a constant k.

    key: random key
    k: constant to add
    """
    x = jax.random.normal(key)  # Standard normal (mean=0, std=1)
    return x + k


# Generate 10 keys
key = jax.random.PRNGKey(123)
keys = jax.random.split(key, 10)

# Apply function with constant k=5.0
results = add_constant_to_random(keys, 5.0)

print("10 Gaussian samples + 5:")
print(results)
print(f"\nMean: {jnp.mean(results):.2f} (should be close to 5.0)")
print(f"Std:  {jnp.std(results):.2f} (should be close to 1.0)")

## 6. JIT Compilation with `jax.jit`

This is JAX's third superpower! `jax.jit` makes your code run **much faster** by compiling it.

### 6.1 What is JIT Compilation?

JIT stands for "Just-In-Time" compilation. Here's what happens:

1. **Normal Python**: Each line is interpreted one at a time → slow
2. **With JIT**: The entire function is compiled to optimized machine code → fast!

The first time you call a JIT-compiled function, there's a small delay for compilation. After that, it's very fast.

### 6.2 Basic Usage

In [ ]:
# Define a function that does some computation
def slow_function(x):
    """A function that does matrix operations 100 times."""
    for _ in range(100):
        x = x @ x.T  # Matrix multiplication with transpose
        x = x / jnp.max(x)  # Normalize to prevent overflow
    return x


# Create a JIT-compiled version
fast_function = jax.jit(slow_function)

print("Created JIT-compiled function!")
print("Type of fast_function:", type(fast_function))

In [ ]:
# Create test data
key = jax.random.PRNGKey(0)
x = jax.random.normal(key, (100, 100))  # 100x100 random matrix

print("Test matrix shape:", x.shape)

In [ ]:
# First, let's "warm up" the JIT function
# The first call includes compilation time, so we do it separately
_ = fast_function(x)
print("JIT compilation done!")

In [ ]:
# Now let's time both versions
import time

# Time the regular function
start = time.time()
_ = slow_function(x)
slow_time = time.time() - start
print(f"Regular function: {slow_time * 1000:.2f} milliseconds")

# Time the JIT function
start = time.time()
_ = fast_function(x)
fast_time = time.time() - start
print(f"JIT function: {fast_time * 1000:.2f} milliseconds")

# Calculate speedup
print(f"\nSpeedup: {slow_time / fast_time:.1f}x faster!")

### 6.3 Using `jit` as a Decorator

The cleanest way to use JIT is as a decorator:

In [ ]:
@jax.jit
def compute_loss(params, x, y):
    """
    A JIT-compiled loss function.
    The @jax.jit decorator automatically compiles this function.
    """
    prediction = params["w"] @ x + params["b"]
    loss = jnp.mean((prediction - y) ** 2)
    return loss


# Set up some test data
params = {"w": jnp.ones((1, 10)), "b": jnp.zeros(1)}
key = jax.random.PRNGKey(0)
key, subkey1 = jax.random.split(key)
key, subkey2 = jax.random.split(key)
x = jax.random.normal(subkey1, (10, 100))
y = jax.random.normal(subkey2, (1, 100))

# Use it just like a normal function!
loss = compute_loss(params, x, y)
print(f"Loss: {loss}")

### 6.4 Important: Static Arguments

Some function arguments can't be compiled (like strings, or options that change the computation structure). Mark these with `static_argnums`:

In [ ]:
@partial(
    jax.jit, static_argnums=(2,)
)  # Argument index 2 is "static" (won't be compiled)
def forward_pass(params, x, activation):
    """
    Forward pass with configurable activation function.

    The 'activation' argument is a string, which can't be compiled,
    so we mark it as static.
    """
    z = params["w"] @ x + params["b"]

    if activation == "relu":
        return jnp.maximum(0, z)
    elif activation == "sigmoid":
        return 1 / (1 + jnp.exp(-z))
    else:
        return z  # No activation (linear)

In [ ]:
# This works even though 'activation' is a string!
output_relu = forward_pass(params, x, "relu")
output_sigmoid = forward_pass(params, x, "sigmoid")

print(f"ReLU output mean: {jnp.mean(output_relu):.4f}")
print(f"Sigmoid output mean: {jnp.mean(output_sigmoid):.4f}")

**When to use `static_argnums`:**
- For string arguments (like activation function names)
- For integers that control the shape of computation (like number of layers)
- For any argument that's not an array and changes the computation structure

## 7. Putting It All Together: Gradient Descent from Scratch

Now let's combine everything we've learned to implement **gradient descent** - the fundamental algorithm behind training neural networks!

### 7.1 The Problem Setup

We'll solve a simple linear regression problem:
- We have data points $(x_i, y_i)$
- We want to find parameters $w$ and $b$ such that $y \approx x \cdot w + b$

**Our approach:**
1. Define a loss function (how wrong our predictions are)
2. Use `jax.grad` to compute gradients of the loss
3. Update parameters to reduce the loss (gradient descent)
4. Use `jax.jit` to make it fast

### 7.2 Step 1: Create Synthetic Data

In [ ]:
# We'll create data that follows y = X @ true_w + true_b + noise

# Set random seed for reproducibility
key = jax.random.PRNGKey(42)

# Generate random input data X (100 samples, 3 features each)
key, subkey = jax.random.split(key)
X = jax.random.normal(subkey, (100, 3))

# Define true parameters (what we want to learn)
true_w = jnp.array([1.0, -2.0, 0.5])  # True weights
true_b = 0.3  # True bias

# Generate y values with some noise
key, subkey = jax.random.split(key)
noise = 0.1 * jax.random.normal(subkey, (100,))
y = X @ true_w + true_b + noise

print("Data shapes:")
print(f"  X: {X.shape} (100 samples, 3 features)")
print(f"  y: {y.shape} (100 target values)")
print(f"\nTrue parameters we want to learn:")
print(f"  w = {true_w}")
print(f"  b = {true_b}")

### 7.3 Step 2: Define the Loss Function

In [ ]:
def loss_fn(params, X, y):
    """
    Mean Squared Error loss function.

    params: dictionary with 'w' (weights) and 'b' (bias)
    X: input data, shape (n_samples, n_features)
    y: target values, shape (n_samples,)

    Returns: scalar loss value
    """
    # Make predictions
    predictions = X @ params["w"] + params["b"]

    # Compute mean squared error
    errors = predictions - y
    loss = jnp.mean(errors**2)

    return loss


# Test the loss function
initial_params = {"w": jnp.zeros(3), "b": 0.0}
initial_loss = loss_fn(initial_params, X, y)
print(f"Initial loss (with zero params): {initial_loss:.4f}")

### 7.4 Step 3: Define the Update Step

This is where `jax.grad`, `jax.jit`, and `jax.tree.map` come together!

In [ ]:
@jax.jit  # Compile for speed
def update_step(params, X, y, learning_rate):
    """
    Perform one step of gradient descent.

    1. Compute the loss and gradients
    2. Update parameters in the direction that reduces the loss

    Returns: (new_params, loss)
    """
    # Compute both loss value and gradients in one call
    loss, grads = jax.value_and_grad(loss_fn)(params, X, y)

    # Update each parameter: new_param = old_param - learning_rate * gradient
    # jax.tree.map applies a function to each leaf of a "tree" (here, our dict)
    new_params = jax.tree.map(
        lambda p, g: p - learning_rate * g,  # Function to apply
        params,  # First tree
        grads,  # Second tree
    )

    return new_params, loss

Let's understand `jax.tree.map`:
- Our `params` is a dictionary: `{"w": array, "b": scalar}`
- Our `grads` has the same structure: `{"w": array, "b": scalar}`
- `jax.tree.map` applies the function to each corresponding pair

In [ ]:
# Let's see what the gradients look like
_, grads = jax.value_and_grad(loss_fn)(initial_params, X, y)
print("Gradients at initial params:")
print(f"  ∂loss/∂w = {grads['w']}")
print(f"  ∂loss/∂b = {grads['b']}")

### 7.5 Step 4: Run the Training Loop!

In [ ]:
# Initialize parameters
params = {"w": jnp.zeros(3), "b": 0.0}

# Hyperparameter
learning_rate = 0.1

# Storage for plotting
losses = []

# Training loop
print("Training...")
print("-" * 40)

for i in range(100):
    params, loss = update_step(params, X, y, learning_rate)
    losses.append(float(loss))

    # Print progress every 20 steps
    if i % 20 == 0:
        print(f"Step {i:3d}: loss = {loss:.6f}")

print("-" * 40)
print("Training complete!")

### 7.6 Step 5: Check the Results

In [ ]:
print("\nComparison of true vs learned parameters:")
print("-" * 40)
print(f"True weights:    {true_w}")
print(f"Learned weights: {params['w']}")
print(f"\nTrue bias:    {true_b}")
print(f"Learned bias: {params['b']:.4f}")

In [ ]:
# Plot the loss curve
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(losses, linewidth=2, color="blue")
ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Loss (Mean Squared Error)", fontsize=12)
ax.set_title("Gradient Descent Training - Loss over Time", fontsize=14)
ax.set_yscale("log")  # Log scale to see the rapid decrease
ax.grid(True, alpha=0.3)

# Add annotations
ax.annotate(
    f"Final loss: {losses[-1]:.4f}",
    xy=(99, losses[-1]),
    xytext=(70, losses[-1] * 3),
    fontsize=10,
    arrowprops=dict(arrowstyle="->", color="gray"),
)

plt.show()

**What we achieved:**
1. Started with zero parameters (loss ≈ 5)
2. Used automatic differentiation to compute gradients
3. Updated parameters step by step
4. Recovered the true parameters almost exactly!

This is the same algorithm (gradient descent) that's used to train all neural networks. The only difference is the complexity of the loss function!

## 8. Summary and Cheat Sheet

Congratulations! You've learned the core concepts of JAX. Here's a summary:

### JAX vs NumPy: Quick Reference

| Task | NumPy | JAX |
|------|-------|-----|
| Import | `import numpy as np` | `import jax.numpy as jnp` |
| Create array | `np.array([1,2,3])` | `jnp.array([1,2,3])` |
| Zeros | `np.zeros((3,3))` | `jnp.zeros((3,3))` |
| Random | `np.random.normal()` | `jax.random.normal(key)` |
| Modify element | `x[0] = 5` | `x = x.at[0].set(5)` |

### JAX's Three Superpowers

| Superpower | Function | What It Does |
|------------|----------|--------------|
| **Auto-differentiation** | `jax.grad(f)` | Computes gradient of `f` |
| **Vectorization** | `jax.vmap(f)` | Makes `f` work on batches |
| **Compilation** | `jax.jit(f)` | Makes `f` run faster |

### Key Patterns to Remember

**1. Computing Gradients:**
```python
f_prime = jax.grad(f)           # Get gradient function
gradient = f_prime(x)            # Compute gradient at x

# Or get both value and gradient:
val, grad = jax.value_and_grad(f)(x)
```

**2. Batching Functions:**
```python
# Apply f to each row of X:
batched_f = jax.vmap(f)
results = batched_f(X)

# Or with decorator:
@partial(jax.vmap, in_axes=(0, None))
def f(x, constant):
    ...
```

**3. Random Numbers:**
```python
key = jax.random.PRNGKey(seed)   # Create key
key, subkey = jax.random.split(key)  # Split before using
x = jax.random.normal(subkey)    # Use subkey
```

**4. JIT Compilation:**
```python
@jax.jit
def fast_function(x):
    ...
```

### What's Next?

Now that you understand JAX fundamentals, you can:
- Build neural networks with **Flax** (a neural network library built on JAX)
- Train models efficiently with **Optax** (an optimization library for JAX)
- Implement advanced algorithms like variational inference, MCMC, and more!
